<h1>4 работа</h1>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

In [2]:
import re

In [3]:
df = pd.read_excel("doc_comment_summary.xlsx", names=["comments", "weights"])
weights = pd.read_excel("full word_rating_after_coding.xlsx", names=["words","weights"])

<h3>предобработка</h3>

In [4]:
df.head()

,comments,weights
0,Украина это часть Руси искусственно отделенная...,-1
1,Как можно говорить об относительно небольшой к...,-1
2,1.2014. а что они со своими поляками сделали?...,0
3,у а фильмы... Зрители любят диковинное. у ме...,0
4,Государство не может сейчас платить больше и м...,-1


In [5]:
df["comments"] = df["comments"].str.lower()

In [6]:
df = df[df['weights'].isin([-2,-1,0,1,2])]

In [7]:
df.dropna(inplace=True)

In [8]:
valid_values = [-2, -1, 0, 1, 2]
df = df[df["weights"].isin(valid_values)]

In [9]:
# Чистим текст
def clean_text(text):
    # Убираем ссылки
    text = re.sub(r'http\S+|www\S+', '', text)
    # Убираем эмодзи
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # Все остальное
    text = re.sub(r'[^а-яА-ЯёЁa-zA-Z ]', '', text)
    return text
df["comments"] = df["comments"].apply(clean_text)

сделаем лемматизацию

In [10]:

import spacy

nlp = spacy.load("ru_core_news_sm")
def lemmatize_text(text):
    # Обработка текста
    doc = nlp(text)
    # Лемматизация каждого слова
    lemmatized_tokens = [token.lemma_ for token in doc]
    # Сборка токенов обратно в строку
    return " ".join(lemmatized_tokens)


In [11]:
df["comments"] = df["comments"].apply(lemmatize_text)

<h3>векторизации</h3>

мешок слов

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_BoW = vectorizer.fit_transform(df['comments'])


tf-idf

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df['comments'])

word2vec

In [14]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in df['comments']]

# Обучаем модель Word2Vec
model = Word2Vec(sentences=tokenized_corpus, vector_size=100, window=5, min_count=1, workers=4)

# Функция для создания вектора текста (усреднение векторов слов)
def text_to_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

# Создаем векторы для текстов
X_wordtovec = np.array([text_to_vector(tokens, model) for tokens in tokenized_corpus])


In [15]:
y = df['weights']
y = y.astype(int)

<h3>Модели</h3>

<h4>Наивный Байес</h4>

In [16]:
from sklearn.naive_bayes import MultinomialNB

mod_NB = MultinomialNB()

In [17]:
# Для BoW
scores = cross_val_score(mod_NB, X_BoW, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.5073818  0.51401869 0.51495327 0.51233645 0.51775701]
Mean accuracy: 0.51


In [18]:
# Для TF-IDF
scores = cross_val_score(mod_NB, X_tfidf, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51859466 0.51794393 0.51757009 0.51700935 0.51906542]
Mean accuracy: 0.52


In [19]:
# Для Word2Vec
scores = cross_val_score(mod_NB, X_wordtovec, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ivans\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\ivans\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ivans\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\naive_bayes.py", line 759, in fit
    self._count(X, Y)
  File "C:\Users\ivans\AppData\Local\Programs\Python\Python312\Lib\site-packages\sk

<h5>наивный Байес не работает с отрицательными числами, а word2vec возвращает еще и отрицательные веса</h5>

<h4>SVM</h4>

In [20]:
from sklearn.svm import SVC

mod_svc = SVC(kernel='linear')

In [21]:
# Для BoW
scores = cross_val_score(mod_svc, X_BoW, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51803401 0.51719626 0.51906542 0.51682243 0.51943925]
Mean accuracy: 0.52


In [22]:
# Для TF-IDF
scores = cross_val_score(mod_svc, X_tfidf, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51859466 0.51794393 0.51981308 0.51570093 0.51869159]
Mean accuracy: 0.52


In [23]:
# Для Word2Vec
scores = cross_val_score(mod_svc, X_wordtovec, y, cv=5, scoring='accuracy') 

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51803401 0.51813084 0.51813084 0.51794393 0.51794393]
Mean accuracy: 0.52


<h4>Деревья</h4>

In [24]:
from sklearn.tree import DecisionTreeClassifier

tree_mod = DecisionTreeClassifier(random_state=42)


In [25]:
# Для BoW
scores = cross_val_score(tree_mod, X_BoW, y, cv=5, scoring='accuracy')

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51317511 0.51327103 0.51364486 0.5117757  0.51514019]
Mean accuracy: 0.51


In [26]:
# Для TF-IDF
scores = cross_val_score(tree_mod, X_tfidf, y, cv=5, scoring='accuracy')

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51429639 0.51308411 0.51364486 0.51271028 0.51551402]
Mean accuracy: 0.51


In [27]:
# Для Word2Vec
scores = cross_val_score(tree_mod, X_wordtovec, y, cv=5, scoring='accuracy')

print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.50682115 0.51214953 0.50953271 0.50747664 0.51439252]
Mean accuracy: 0.51


<h4>Ансамбли</h4>

используем градиентный бустинг

In [28]:
from sklearn.ensemble import GradientBoostingClassifier

In [29]:
grbust_mod = GradientBoostingClassifier(n_estimators=100, random_state=42)

In [30]:
# Для BoW
scores = cross_val_score(grbust_mod, X_BoW, y, cv=5, scoring='accuracy')

# Вывод результатов
print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51840777 0.51775701 0.51943925 0.51626168 0.51981308]
Mean accuracy: 0.52


In [31]:
# Для TF-IDF
scores = cross_val_score(grbust_mod, X_tfidf, y, cv=5, scoring='accuracy')

# Вывод результатов
print(f'Accuracy for each fold: {scores}')
print(f'Mean accuracy: {scores.mean():.2f}')

Accuracy for each fold: [0.51896842 0.51794393 0.5188785  0.51757009 0.51925234]
Mean accuracy: 0.52


In [32]:
# Для Word2Vec
sc = cross_val_score(grbust_mod, X_wordtovec, y, cv=5, scoring='accuracy')

# Вывод результатов
print(f'Accuracy for each fold: {sc}')
print(f'Mean accuracy: {sc.mean():.2f}')

Accuracy for each fold: [0.51728649 0.51514019 0.51831776 0.51327103 0.51757009]
Mean accuracy: 0.52


ни один метод векторизации не выделился особым результатом, далее будет использован только метод Word2Vec

<h3>А теперь тональный словарик</h3>

In [33]:
weights.head()

,words,weights
0,абажур,0
1,абажур,-1
2,абориген,-1
3,абориген,-1
4,абориген,0


сделаем агригацию словаря

In [34]:
aggregated_dict = weights.groupby('words')['weights'].mean().reset_index()

In [35]:
aggregated_dict.head()

,words,weights
0,абажур,-0.500000
1,абориген,-0.666667
2,аборт,-0.750000
3,абортивный,-0.250000
4,абсолютный,0.000000


In [36]:
def calculate_text_score(text, aggregated_dict):
    words = text.split()
    scores = []
    for word in words:
        if word in aggregated_dict['words'].values:
            score = aggregated_dict[aggregated_dict['words'] == word]['score'].values[0]
            scores.append(score)
    return sum(scores) / len(scores) if scores else 0

df['text_score'] = df['comments'].apply(lambda x: calculate_text_score(x, aggregated_dict))

In [37]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['comments'])
y = df['weights']

In [43]:
y.head()

0   -1
1   -1
2    0
3    0
4   -1
Name: weights, dtype: int32

In [42]:
y = y.astype(int)

In [40]:
import scipy.sparse as sp

# Преобразуем text_score в разреженную матрицу
text_score_sparse = sp.csr_matrix(df['text_score'].values.reshape(-1, 1))

# Объединяем признаки
X_combined = sp.hstack([X, text_score_sparse])

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Обучаем модель (например, RandomForestClassifier)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Оцениваем модель
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')

Accuracy: 0.5096243692767707
